# Treino U-Net no Google Colab (GPU)

Pré-treino (Inria, binário) + fine-tuning (chips multiclasse com as tuas anotações).

**Antes de correr:**
1. **Runtime → Change runtime type → GPU** (T4 ou melhor).
2. **Projeto:** o notebook clona https://github.com/phmotad/roofArea.git na célula abaixo.
3. **Dados:** na célula indicada, faz upload de `chips_multiclass.zip` (pasta com `images/` e `masks/`) ou usa a pasta se já estiver no ambiente.

In [1]:
# Verificar GPU
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if out.returncode == 0:
    print('GPU:', out.stdout.strip())
else:
    print('Nenhuma GPU detectada. Menu Runtime → Change runtime type → GPU')

GPU: Tesla T4, 15360 MiB


In [ ]:
# Se o projeto já estiver presente (Cursor Colab), não faz nada.
# Senão: faz upload de um zip do projeto (raiz com src/, scripts/, pyproject.toml ou setup.py).

import os
import zipfile

if os.path.isfile('scripts/train_unet.py') and os.path.isdir('src'):
    print('Projeto já presente.')
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = [k for k in uploaded if k.endswith('.zip')][0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('.')
    root = '.'
    if not os.path.isfile('scripts/train_unet.py'):
        for name in os.listdir('.'):
            if os.path.isfile(os.path.join(name, 'scripts', 'train_unet.py')):
                root = name
                break
    if root != '.':
        os.chdir(root)
        print('Entrei em', root)
    print('Projeto pronto.')

**Como criar o zip:** na pasta do projeto (no PC), seleciona a raiz (pastas `src/`, `scripts/`, ficheiro `pyproject.toml` ou `setup.py`) e comprime. No Colab, faz upload desse zip na célula acima.

In [3]:
# Instalar dependências (só as necessárias para treino)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q pillow numpy opencv-python-headless scikit-image huggingface_hub
!pip install -q -e .

In [4]:
# Descarregar e converter dataset Inria (pré-treino binário)
!python -m scripts.download_inria_dataset --output_dir ./dados_inria

In [5]:
# Teus chips multiclasse: se já existir ./chips_multiclass (ex.: Cursor Colab), não faz nada.
# Senão, faz upload de chips_multiclass.zip (pasta com images/ e masks/).
import os
import zipfile

if os.path.isdir('chips_multiclass') and os.path.isdir('chips_multiclass/images'):
    print('chips_multiclass já presente.')
else:
    from google.colab import files
    uploaded = files.upload()  # escolhe chips_multiclass.zip
    zip_name = [k for k in uploaded if k.endswith('.zip')][0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('.')
    print('Extraído:', zip_name)

**Alternativa: Google Drive** — se tiveres a pasta no Drive, monta e copia:
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r '/content/drive/MyDrive/caminho/para/chips_multiclass' ./chips_multiclass
```

In [ ]:
# Pré-treino (binário, Inria) — GPU
!python -u -m scripts.train_unet \
  --data_dir ./dados_inria \
  --output ./models/unet_roof_pretrain.pt \
  --num_classes 1 \
  --size 512 512 \
  --epochs 30 \
  --device cuda

In [ ]:
# Fine-tuning (multiclasse, teus chips) — GPU
!python -u -m scripts.train_unet \
  --data_dir ./chips_multiclass \
  --output ./models/unet_roof_multiclass.pt \
  --num_classes 5 \
  --size 512 512 \
  --epochs 50 \
  --pretrain ./models/unet_roof_pretrain.pt \
  --device cuda

## Teste: inferência com o modelo treinado
Carrega o modelo, corre numa imagem de teste e mostra a máscara + área estimada.

In [ ]:
# Teste: imagem do telhado + tamanho (área estimada)
import sys
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

if Path('src').is_dir():
    sys.path.insert(0, '.')
    sys.path.insert(0, 'src')

import torch
from roof_api.segmentation.unet_model import load_unet, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_unet('models/unet_roof_multiclass.pt', device, num_classes=5)

images_dir = Path('chips_multiclass/images')
test_images = sorted(images_dir.glob('*.png'))[:3]
if not test_images:
    test_images = list(Path('.').glob('**/images/*.png'))[:3]

METROS_POR_PIXEL = 0.10

for path in test_images:
    rgb = np.array(Image.open(path).convert('RGB'), dtype=np.uint8)
    prob = predict(model, rgb, device)
    mask = (prob > 0.6).astype(np.uint8)
    area_px = int(mask.sum())
    area_m2 = area_px * (METROS_POR_PIXEL ** 2)
    overlay = rgb.copy().astype(np.float32)
    overlay[mask > 0] = overlay[mask > 0] * 0.5 + np.array([255, 80, 80], dtype=np.float32) * 0.5
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    print(f'{path.name}: área estimada {area_m2:.1f} m² ({area_px} px, {METROS_POR_PIXEL} m/px)')
    from IPython.display import display
    display(Image.fromarray(overlay))

In [ ]:
# Descarregar o modelo treinado para o teu PC
from google.colab import files
files.download('models/unet_roof_multiclass.pt')

Opcional: guardar no Google Drive em vez de download:
```python
!cp models/unet_roof_multiclass.pt '/content/drive/MyDrive/'
```